# 05. Feature Engineering

## Objective

This notebook prepares the feature layer for the credit risk model.

It builds on the time-based split from Notebook 04 and focuses on:

- reviewing candidate modelling variables
- removing technical / post-outcome / non-modelling columns
- creating derived credit-risk features
- handling date-based variables
- creating missing-value indicators where useful
- applying simple outlier capping rules based on the training sample only
- exporting engineered train / validation / OOT datasets


In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Paths

Input is taken from Notebook 04:

`../data/processed/04.time_split_and_sampling_strategy/`

Outputs are saved to:

`../data/processed/05.feature_engineering/`

In [ ]:
PROJECT_ROOT = Path("..")

INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "04.time_split_and_sampling_strategy"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "05.feature_engineering"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_path = INPUT_DIR / "train.pkl"
validation_path = INPUT_DIR / "validation.pkl"
oot_path = INPUT_DIR / "oot_test.pkl"

for path in [train_path, validation_path, oot_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected input file: {path}")

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Load train / validation / OOT datasets

In [ ]:
train = pd.read_pickle(train_path)
validation = pd.read_pickle(validation_path)
oot_test = pd.read_pickle(oot_path)

print(f"Train shape:      {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"OOT test shape:   {oot_test.shape}")

display(train.head())

## 3. Basic checks

Expected target:

- `target_bad = 1`: bad / charged off
- `target_bad = 0`: good / fully paid

Expected time column:

- `issue_d`: loan origination / disbursement date

In [ ]:
TARGET_COL = "target_bad"
DATE_COL = "issue_d"

for dataset_name, data in {"train": train, "validation": validation, "oot_test": oot_test}.items():
    if TARGET_COL not in data.columns:
        raise ValueError(f"{TARGET_COL} missing from {dataset_name}.")
    if DATE_COL not in data.columns:
        raise ValueError(f"{DATE_COL} missing from {dataset_name}.")

for data in [train, validation, oot_test]:
    data[DATE_COL] = pd.to_datetime(data[DATE_COL], errors="coerce")
    if "earliest_cr_line" in data.columns:
        data["earliest_cr_line"] = pd.to_datetime(data["earliest_cr_line"], errors="coerce")

summary = pd.DataFrame({
    "sample": ["train", "validation", "oot_test"],
    "rows": [len(train), len(validation), len(oot_test)],
    "bad_rate": [train[TARGET_COL].mean(), validation[TARGET_COL].mean(), oot_test[TARGET_COL].mean()],
    "issue_min": [train[DATE_COL].min(), validation[DATE_COL].min(), oot_test[DATE_COL].min()],
    "issue_max": [train[DATE_COL].max(), validation[DATE_COL].max(), oot_test[DATE_COL].max()],
})

summary

In [ ]:
train["earliest_cr_line"].head()


## 4. Candidate feature review

At this stage, we separate columns into:

1. **Core modelling variables** — available at or close to origination
2. **Derived variables** — engineered in this notebook
3. **Excluded variables** — target, technical columns, split labels, post-outcome/leakage variables, raw high-cardinality text fields


In [ ]:
all_cols = train.columns.tolist()

technical_cols = [
    TARGET_COL, "loan_status", "sample", "issue_year", "issue_quarter",
    "issue_month", "issue_quarter_period",
]

post_outcome_or_leakage_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag",
    "debt_settlement_flag_date", "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
    "out_prncp", "out_prncp_inv",
]

raw_high_cardinality_cols = [
    "id", "member_id", "url", "desc", "title", "emp_title", "zip_code",
    "addr_state",  # excluded from baseline scorecard to avoid geography/policy complexity
]

excluded_cols = sorted(set(
    [c for c in technical_cols if c in all_cols]
    + [c for c in post_outcome_or_leakage_cols if c in all_cols]
    + [c for c in raw_high_cardinality_cols if c in all_cols]
))

candidate_base_features = [c for c in all_cols if c not in excluded_cols]

feature_review = pd.DataFrame({
    "variable": all_cols,
    "role": ["excluded" if c in excluded_cols else "candidate_base_feature" for c in all_cols],
    "dtype": [str(train[c].dtype) for c in all_cols],
    "missing_pct_train": [train[c].isna().mean() for c in all_cols],
    "nunique_train": [train[c].nunique(dropna=False) for c in all_cols],
})

feature_review.sort_values(["role", "variable"]).head(100)

## 5. Feature engineering functions

All feature engineering logic is wrapped in functions so the exact same transformations are applied to train, validation and OOT.

> Rules and caps are learned only from the training sample and then applied unchanged to validation and OOT.

In [ ]:
def parse_term_months(s):
    if pd.isna(s):
        return np.nan
    text = str(s)
    digits = "".join(ch for ch in text if ch.isdigit())
    return float(digits) if digits else np.nan


def engineer_features(data):
    data = data.copy()
    data["issue_d"] = pd.to_datetime(data["issue_d"], errors="coerce")
    
    if "earliest_cr_line" in data.columns:
        data["earliest_cr_line"] = pd.to_datetime(data["earliest_cr_line"], errors="coerce")
        data["credit_history_months"] = (
            (data["issue_d"].dt.year - data["earliest_cr_line"].dt.year) * 12
            + (data["issue_d"].dt.month - data["earliest_cr_line"].dt.month)
        )
        data["credit_history_years"] = data["credit_history_months"] / 12
        data["credit_history_missing_flag"] = data["earliest_cr_line"].isna().astype(int)
    
    if "term" in data.columns:
        data["term_months"] = data["term"].apply(parse_term_months)
        data["is_60m_term"] = (data["term_months"] == 60).astype(int)
    
    if {"fico_range_low", "fico_range_high"}.issubset(data.columns):
        data["fico_mid"] = (data["fico_range_low"] + data["fico_range_high"]) / 2
    elif "fico_range_high" in data.columns:
        data["fico_mid"] = data["fico_range_high"] - 5
    
    if {"loan_amnt", "annual_inc"}.issubset(data.columns):
        data["loan_to_income"] = np.where(data["annual_inc"] > 0, data["loan_amnt"] / data["annual_inc"], np.nan)
        data["annual_inc_missing_flag"] = data["annual_inc"].isna().astype(int)
    
    if {"installment", "annual_inc"}.issubset(data.columns):
        data["installment_to_monthly_income"] = np.where(
            data["annual_inc"] > 0,
            data["installment"] / (data["annual_inc"] / 12),
            np.nan,
        )
    
    if "revol_util" in data.columns:
        data["revol_util_missing_flag"] = data["revol_util"].isna().astype(int)
        data["high_revol_util_flag"] = (data["revol_util"] >= 80).astype(int)
    
    if "dti" in data.columns:
        data["dti_missing_flag"] = data["dti"].isna().astype(int)
        data["high_dti_flag"] = (data["dti"] >= 35).astype(int)
    
    if "delinq_2yrs" in data.columns:
        data["has_recent_delinq_flag"] = (data["delinq_2yrs"].fillna(0) > 0).astype(int)
    
    if "inq_last_6mths" in data.columns:
        data["has_recent_inquiry_flag"] = (data["inq_last_6mths"].fillna(0) > 0).astype(int)
    
    if "pub_rec" in data.columns:
        data["has_public_record_flag"] = (data["pub_rec"].fillna(0) > 0).astype(int)
    
    if "pub_rec_bankruptcies" in data.columns:
        data["has_bankruptcy_flag"] = (data["pub_rec_bankruptcies"].fillna(0) > 0).astype(int)
    
    if "mort_acc" in data.columns:
        data["has_mortgage_account_flag"] = (data["mort_acc"].fillna(0) > 0).astype(int)
    
    if "emp_length" in data.columns:
        emp = data["emp_length"].astype(str).str.lower()
        data["emp_length_missing_flag"] = data["emp_length"].isna().astype(int)
        data["emp_length_num"] = np.nan
        data.loc[emp.str.contains("< 1", na=False), "emp_length_num"] = 0
        data.loc[emp.str.contains("10+", regex=False, na=False), "emp_length_num"] = 10
        extracted = emp.str.extract(r"(\d+)")[0]
        mask = data["emp_length_num"].isna() & extracted.notna()
        data.loc[mask, "emp_length_num"] = extracted[mask].astype(float)
    
    data["issue_year_fe"] = data["issue_d"].dt.year
    data["issue_quarter_fe"] = data["issue_d"].dt.quarter
    
    return data


train_fe = engineer_features(train)
validation_fe = engineer_features(validation)
oot_test_fe = engineer_features(oot_test)

print(train_fe.shape, validation_fe.shape, oot_test_fe.shape)

## 6. Outlier capping

Selected numeric features are capped at the 1st and 99th percentile based on the **training sample only**.

The same thresholds are then applied to validation and OOT.

In [ ]:
candidate_cap_vars = [
    "annual_inc", "dti", "revol_util", "revol_bal", "loan_amnt", "installment",
    "loan_to_income", "installment_to_monthly_income", "credit_history_months",
    "credit_history_years", "open_acc", "total_acc", "inq_last_6mths",
    "delinq_2yrs", "pub_rec", "mort_acc",
]

cap_vars = [c for c in candidate_cap_vars if c in train_fe.columns and pd.api.types.is_numeric_dtype(train_fe[c])]

cap_rules = {}

for col in cap_vars:
    lower = train_fe[col].quantile(0.01)
    upper = train_fe[col].quantile(0.99)
    if pd.isna(lower) or pd.isna(upper) or lower == upper:
        continue
    cap_rules[col] = {"lower_p01": lower, "upper_p99": upper}

cap_rules_df = pd.DataFrame(cap_rules).T.reset_index().rename(columns={"index": "variable"})
cap_rules_df

In [ ]:
def apply_capping(data, cap_rules):
    data = data.copy()
    for col, rules in cap_rules.items():
        if col in data.columns:
            data[f"{col}_capped"] = data[col].clip(lower=rules["lower_p01"], upper=rules["upper_p99"])
    return data

train_fe = apply_capping(train_fe, cap_rules)
validation_fe = apply_capping(validation_fe, cap_rules)
oot_test_fe = apply_capping(oot_test_fe, cap_rules)

print(train_fe.shape, validation_fe.shape, oot_test_fe.shape)

## 7. Final candidate feature list

The output keeps original non-excluded candidate variables plus engineered features.

In [ ]:
engineered_cols = sorted([c for c in train_fe.columns if c not in train.columns])
excluded_cols_final = sorted(set([c for c in excluded_cols if c in train_fe.columns]))

candidate_features = [c for c in train_fe.columns if c not in excluded_cols_final and c != TARGET_COL]

raw_date_cols = [c for c in ["issue_d", "earliest_cr_line"] if c in candidate_features]

candidate_model_features = [c for c in candidate_features if c not in raw_date_cols]

all_fe_cols = sorted(set(train_fe.columns))
feature_inventory = pd.DataFrame({
    "variable": all_fe_cols,
    "is_original": [c in train.columns for c in all_fe_cols],
    "is_engineered": [c in engineered_cols for c in all_fe_cols],
    "is_excluded": [c in excluded_cols_final for c in all_fe_cols],
    "is_candidate_model_feature": [c in candidate_model_features for c in all_fe_cols],
    "dtype": [str(train_fe[c].dtype) for c in all_fe_cols],
    "missing_pct_train": [train_fe[c].isna().mean() for c in all_fe_cols],
    "nunique_train": [train_fe[c].nunique(dropna=False) for c in all_fe_cols],
})

print(f"Original columns: {len(train.columns)}")
print(f"Engineered columns: {len(engineered_cols)}")
print(f"Excluded columns: {len(excluded_cols_final)}")
print(f"Candidate model features: {len(candidate_model_features)}")

feature_inventory.sort_values(["is_candidate_model_feature", "variable"], ascending=[False, True]).head(100)

## 8. Missingness comparison after feature engineering

In [ ]:
def missingness_table(data, sample_name):
    return (
        data[candidate_model_features]
        .isna()
        .mean()
        .reset_index()
        .rename(columns={"index": "variable", 0: "missing_pct"})
        .assign(sample=sample_name)
    )

missingness_summary = pd.concat([
    missingness_table(train_fe, "train"),
    missingness_table(validation_fe, "validation"),
    missingness_table(oot_test_fe, "oot_test"),
], ignore_index=True)

missingness_pivot = missingness_summary.pivot_table(
    index="variable", columns="sample", values="missing_pct", aggfunc="first"
).reset_index()

missingness_pivot["max_missing_pct"] = missingness_pivot[["train", "validation", "oot_test"]].max(axis=1)

missingness_pivot.sort_values("max_missing_pct", ascending=False).head(50)

## 9. Quick univariate check for engineered features

In [ ]:
engineered_numeric_cols = [c for c in engineered_cols if c in train_fe.columns and pd.api.types.is_numeric_dtype(train_fe[c])]

def quick_numeric_bad_rate(data, variable, target_col=TARGET_COL, bins=10):
    temp = data[[variable, target_col]].copy().dropna(subset=[variable])
    if temp.empty:
        return pd.DataFrame()

    if temp[variable].nunique() < 3:
        out = (
            temp.groupby(variable)
            .agg(total_accounts=(target_col, "size"), bad_count=(target_col, "sum"), observed_bad_rate=(target_col, "mean"))
            .reset_index()
        )
        out.insert(0, "variable", variable)
        return out

    try:
        temp["bin"] = pd.qcut(temp[variable], q=bins, duplicates="drop")
    except ValueError:
        temp["bin"] = pd.cut(temp[variable], bins=bins)

    out = (
        temp.groupby("bin", observed=False)
        .agg(
            total_accounts=(target_col, "size"),
            bad_count=(target_col, "sum"),
            observed_bad_rate=(target_col, "mean"),
            min_value=(variable, "min"),
            max_value=(variable, "max"),
        )
        .reset_index()
    )
    out.insert(0, "variable", variable)
    return out

engineered_univariate_tables = {}

for col in engineered_numeric_cols:
    engineered_univariate_tables[col] = quick_numeric_bad_rate(train_fe, col)

for col in ["fico_mid", "loan_to_income", "credit_history_months", "installment_to_monthly_income"]:
    if col in engineered_univariate_tables:
        print(f"\n{col}")
        display(engineered_univariate_tables[col])

In [ ]:
selected_engineered_features = [
    "fico_mid", "loan_to_income", "installment_to_monthly_income",
    "credit_history_months", "revol_util_capped", "dti_capped",
]

for col in selected_engineered_features:
    if col not in train_fe.columns:
        continue

    table = quick_numeric_bad_rate(train_fe, col)
    if table.empty or "bin" not in table.columns:
        continue

    x_labels = table["bin"].astype(str)
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.bar(x_labels, table["total_accounts"], alpha=0.75)
    ax1.set_ylabel("Total accounts")
    ax1.set_xlabel(col)
    ax1.tick_params(axis="x", rotation=45)
    ax1.grid(axis="y", linestyle="--", alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x_labels, table["observed_bad_rate"], color="red", marker="o", linewidth=2.5)
    ax2.set_ylabel("Observed bad rate", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    plt.title(f"{col}: Volume and Observed Bad Rate - Train")
    plt.tight_layout()
    plt.show()

## 10. Save outputs

This notebook saves engineered train / validation / OOT datasets, candidate feature lists, capping rules and Excel/PKL artifacts.

In [ ]:
train_fe.to_pickle(OUTPUT_DIR / "train_fe.pkl")
validation_fe.to_pickle(OUTPUT_DIR / "validation_fe.pkl")
oot_test_fe.to_pickle(OUTPUT_DIR / "oot_test_fe.pkl")

with open(OUTPUT_DIR / "candidate_model_features.pkl", "wb") as f:
    pickle.dump(candidate_model_features, f)

artifacts = {
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "excluded_cols": excluded_cols_final,
    "engineered_cols": engineered_cols,
    "candidate_model_features": candidate_model_features,
    "feature_review": feature_review,
    "feature_inventory": feature_inventory,
    "cap_rules": cap_rules,
    "cap_rules_df": cap_rules_df,
    "missingness_summary": missingness_summary,
    "missingness_pivot": missingness_pivot,
    "engineered_univariate_tables": engineered_univariate_tables,
}

with open(OUTPUT_DIR / "feature_engineering_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

excel_path = OUTPUT_DIR / "feature_engineering_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    feature_review.to_excel(writer, sheet_name="feature_review", index=False)
    feature_inventory.to_excel(writer, sheet_name="feature_inventory", index=False)
    cap_rules_df.to_excel(writer, sheet_name="capping_rules", index=False)
    missingness_pivot.to_excel(writer, sheet_name="missingness", index=False)
    pd.DataFrame({"candidate_model_features": candidate_model_features}).to_excel(writer, sheet_name="candidate_features", index=False)

    for col, table in engineered_univariate_tables.items():
        if not table.empty:
            sheet_name = f"univar_{col}"[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'train_fe.pkl'}")
print(f"- {OUTPUT_DIR / 'validation_fe.pkl'}")
print(f"- {OUTPUT_DIR / 'oot_test_fe.pkl'}")
print(f"- {OUTPUT_DIR / 'candidate_model_features.pkl'}")
print(f"- {OUTPUT_DIR / 'feature_engineering_artifacts.pkl'}")
print(f"- {excel_path}")

## 11. Conclusions

The feature layer is now prepared for WOE/IV analysis.

Main transformations performed:

- created credit-history length features
- created FICO midpoint
- created loan affordability ratios
- created delinquency, inquiry, public record and bankruptcy flags
- extracted numeric employment length
- created missing-value flags for selected important variables
- capped selected numeric variables using training-sample 1st/99th percentiles

